[Reference](https://levelup.gitconnected.com/i-tuned-a-7b-model-that-outperforms-gpt-4-heres-how-you-can-too-96e7bede8899)

# Step 1: Environment Setup

In [1]:
# Install required libraries
!pip install unsloth trl peft accelerate bitsandbytes

# Verify GPU access
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 137.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 141.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Step 2: Load and Prepare Your Base Model

In [2]:
from unsloth import FastLanguageModel

# Load the base model with 4-bit quantization for efficiency
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,  # Auto-detect optimal type
    load_in_4bit=True,  # Reduces memory usage by 75%
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.8: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Step 3: Create Your Training Dataset

In [3]:
import json
from datasets import Dataset

# Sample training data (you'll want 500-5000 examples for production)
training_data = [
    {
        "input": "Extract product info from: <div class='product'><h2>iPhone 15</h2><span class='price'>$999</span><span class='category'>Electronics</span></div>",
        "output": {"name": "iPhone 15", "price": "$999", "category": "Electronics"}
    },
    {
        "input": "Extract product info from: <div class='product'><h2>Nike Air Max</h2><span class='price'>$129</span><span class='category'>Footwear</span></div>",
        "output": {"name": "Nike Air Max", "price": "$129", "category": "Footwear"}
    }
    # Add more examples...
]

def create_training_prompt(sample):
    """Convert raw data into properly formatted training prompts"""
    return f"""### Task: Extract product information from HTML
### Input: {sample['input']}
### Output: {json.dumps(sample['output'], indent=2)}
### Status: Complete<|endoftext|>"""

# Create the dataset
formatted_prompts = [create_training_prompt(item) for item in training_data]
dataset = Dataset.from_dict({"text": formatted_prompts})

# Step 4: Configure LoRA for Efficient Training

In [4]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # Rank - balance between performance and efficiency
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",     # MLP layers
    ],
    lora_alpha=64,  # Scaling factor
    lora_dropout=0.05,  # Prevent overfitting
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory optimization
    random_state=3407,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.10.8 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


# Step 5: Train Your Model

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Set up the trainer with beginner-friendly settings
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,  # Adjust based on your GPU memory
        gradient_accumulation_steps=4,  # Effective batch size = 2 * 4 = 8
        warmup_steps=20,
        num_train_epochs=3,  # Usually 1-5 epochs is enough
        learning_rate=2e-4,
        fp16=True,  # Use mixed precision for efficiency
        logging_steps=10,
        optim="adamw_8bit",  # Memory-efficient optimizer
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="product_extractor_model",
        save_strategy="epoch",
        # evaluation_strategy="no",  # Simplify for beginners
    ),
)
# Start training (typically takes 10-30 minutes)
trainer.train()

# Step 6: Test Your Model

In [9]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test with a sample input
test_input = "Extract product info from: <div class='product'><h2>MacBook Pro</h2><span class='price'>$1,999</span><span class='category'>Computers</span></div>"

# Format the input
messages = [{"role": "user", "content": test_input}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.1,  # Lower = more consistent
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# Decode and display result
response = tokenizer.batch_decode(outputs)[0]
print(response)

# Step 7: Deploy with Ollama

In [10]:
# Export to GGUF format for Ollama
model.save_pretrained_gguf(
    "product_extractor_final",
    tokenizer,
    quantization_method="q4_k_m"  # Good balance of size and quality
)

# Download the model file
from google.colab import files
import os

gguf_files = [f for f in os.listdir("product_extractor_final") if f.endswith(".gguf")]
if gguf_files:
    files.download(f"product_extractor_final/{gguf_files[0]}")

```
# Create the model in Ollama
ollama create product-extractor -f Modelfile

# Run your custom model
ollama run product-extractor
```